## IMAGE INFERENCE

In [1]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv
from PIL import Image
import io

# Load environment variables from .env file
load_dotenv()

def process_image_with_nova_arn_fixed(
    image_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this image content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process image using AWS Bedrock Nova Pro with ARN profile - Fixed version
    Handles MIME type issues and image validation better
    
    Args:
        image_path: Path to the image file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for image analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Image file validation
        if not os.path.exists(image_path):
            return f"Error: Image file not found: {image_path}"
        
        file_size = os.path.getsize(image_path)
        image_name = os.path.basename(image_path)
        
        print(f"Processing image: {image_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Image too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Validate and potentially fix image using PIL
        try:
            with Image.open(image_path) as img:
                print(f"Original image format: {img.format}")
                print(f"Original image mode: {img.mode}")
                print(f"Original image size: {img.size}")
                
                # Convert to RGB if necessary (Nova Pro works best with RGB)
                if img.mode != 'RGB':
                    print(f"Converting from {img.mode} to RGB")
                    img = img.convert('RGB')
                
                # Resize if too large (Nova Pro has limits)
                max_size = 2048
                if max(img.size) > max_size:
                    print(f"Resizing image from {img.size} to fit {max_size}px limit")
                    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
                
                # Save to bytes buffer in PNG format for consistency
                buffer = io.BytesIO()
                img.save(buffer, format='PNG')
                image_bytes = buffer.getvalue()
                
                print(f"Processed image bytes length: {len(image_bytes)}")
                print(f"Final image format: PNG")
                print(f"Final image size: {img.size}")
                
        except Exception as e:
            print(f"PIL processing failed: {str(e)}")
            # Fallback to original file
            with open(image_path, 'rb') as image_file:
                image_bytes = image_file.read()
        
        # Encode to base64
        print("Encoding image to base64...")
        image_b64 = base64.b64encode(image_bytes).decode('utf-8')
        
        print(f"Base64 string length: {len(image_b64)}")
        
        # Prepare the message content - use PNG format for consistency
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",  # Always use PNG for consistency
                            "source": {
                                "bytes": image_b64
                            }
                        }
                    },
                    {"text": prompt}
                ]
            }
        ]
        
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Try using converse method first
        try:
            print("Trying converse method for image analysis...")
            response = bedrock.converse(
                modelId="amazon.nova-pro-v1:0",
                messages=messages,
                inferenceConfig=inference_config
            )
            
            # Extract response text
            result = response['output']['message']['content'][0]['text']
            print("Image analysis completed successfully with converse method")
            return result
            
        except Exception as e:
            print(f"converse method failed: {str(e)}")
            print("Falling back to invoke_model method...")
            
            # Fallback to invoke_model method
            try:
                print("Trying invoke_model method...")
                response = bedrock.invoke_model(
                    modelId="amazon.nova-pro-v1:0",
                    body=json.dumps({
                        "messages": messages,
                        "inferenceConfig": inference_config
                    }),
                    contentType="application/json"
                )
                
                # Parse response
                response_body = json.loads(response['body'].read())
                result = response_body['output']['message']['content'][0]['text']
                print("Image analysis completed successfully with invoke_model method")
                return result
                
            except Exception as e2:
                print(f"invoke_model also failed: {str(e2)}")
                return f"Both API methods failed. Image processing errors: {str(e)}, {str(e2)}"
        
    except FileNotFoundError:
        return f"Error: Image file not found: {image_path}"
    except Exception as e:
        return f"Error processing image: {str(e)}"

def validate_arn_format(arn: str) -> bool:
    """Validate ARN format"""
    return arn.startswith("arn:aws:bedrock:") and ("inference-profile" in arn or "model-access-policy" in arn)

# Test the fixed image processing
if __name__ == "__main__":
    # Configuration
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    
    # Test with the new stock chart image
    image_file = "data/images/stock_chart.png"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    print("="*60)
    print("TESTING FIXED IMAGE PROCESSING")
    print("="*60)
    
    if os.path.exists(image_file):
        print(f"Testing with: {image_file}")
        image_result = process_image_with_nova_arn_fixed(
            image_file,
            profile_arn,
            "Analyze this financial chart image and describe what you see. Focus on trends, patterns, and any notable data points."
        )
        print(f"\nImage Analysis Result:")
        print("-" * 50)
        print(image_result)
    else:
        print(f"Image file not found: {image_file}")
        
        # Try with the fixed red square
        fallback_image = "data/images/test_red_square_fixed.png"
        if os.path.exists(fallback_image):
            print(f"Testing with fallback: {fallback_image}")
            image_result = process_image_with_nova_arn_fixed(
                fallback_image,
                profile_arn,
                "Describe this simple image."
            )
            print(f"\nImage Analysis Result:")
            print("-" * 50)
            print(image_result)
        else:
            print("No suitable test images found")


TESTING FIXED IMAGE PROCESSING
Testing with: data/images/stock_chart.png
Processing image: stock_chart.png
File size: 50.98 KB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Original image format: PNG
Original image mode: RGBA
Original image size: (1182, 880)
Converting from RGBA to RGB
Processed image bytes length: 49304
Final image format: PNG
Final image size: (1182, 880)
Encoding image to base64...
Base64 string length: 65740
Sending request to Bedrock Nova Pro...
Trying converse method for image analysis...
converse method failed: An error occurred (ValidationException) when calling the Converse operation: The model returned the following errors: The detected file MIME type text/plain does not match the expected type image/png. Reformat your input and try again.
Falling back to invoke_model method...
Trying invoke_model method...
Image analysis completed successfully with invoke_model method

Image Analysis Result:
------

## VIDEO INFERENCE

In [2]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

def process_video_with_nova_arn(
    video_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this video content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process video using AWS Bedrock Nova Pro with ARN profile
    
    Args:
        video_path: Path to the video file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for video analysis
        aws_profile: Optional AWS profile name (instead of hardcoded keys)
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Video file validation
        if not os.path.exists(video_path):
            return f"Error: Video file not found: {video_path}"
        
        file_size = os.path.getsize(video_path)
        video_name = os.path.basename(video_path)
        
        print(f"Processing video: {video_name}")
        print(f"File size: {file_size / (1024*1024):.2f} MB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Video too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Read and encode video file
        print("Encoding video to base64...")
        with open(video_path, 'rb') as video_file:
            video_bytes = video_file.read()
            video_b64 = base64.b64encode(video_bytes).decode('utf-8')
        
        print(f"Video bytes length: {len(video_bytes)}")
        print(f"Base64 string length: {len(video_b64)}")
        print(f"Base64 starts with: {video_b64[:50]}...")
        
        # Determine video format from file extension
        file_ext = os.path.splitext(video_path)[1].lower()
        format_map = {
            '.mp4': 'mp4',
            '.mov': 'mov',
            '.avi': 'avi',
            '.webm': 'webm'
        }
        video_format = format_map.get(file_ext, 'mp4')
        
        print(f"Detected video format: {video_format}")
        print(f"File extension: {file_ext}")
        
        # Prepare the message content
        messages = [
        {
            "role": "user",
            "content": [
                {
                    "video": {
                        "format": video_format,  # Use the detected format
                        "source": {
                            "bytes": video_b64
                        }
                    }
                },
                {"text": prompt}
            ]
        }
    ]
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Try using invoke_model instead of converse for better video handling
        try:
            print("Trying invoke_model method...")
            response = bedrock.invoke_model(
                modelId="amazon.nova-pro-v1:0",
                body=json.dumps({
                    "messages": messages,
                    "inferenceConfig": inference_config
                }),
                contentType="application/json"
            )
            
            # Parse response
            response_body = json.loads(response['body'].read())
            result = response_body['output']['message']['content'][0]['text']
            
        except Exception as e:
            print(f"invoke_model failed: {str(e)}")
            print("Falling back to converse method...")
            
            # Fallback to converse method
            response = bedrock.converse(
                modelId="amazon.nova-pro-v1:0",
                messages=messages,
                inferenceConfig=inference_config
            )
            
            # Extract response text
            result = response['output']['message']['content'][0]['text']
        
        print("Analysis completed successfully")
        return result
        
    except FileNotFoundError:
        return f"Error: Video file not found: {video_path}"
    except Exception as e:
        return f"Error processing video: {str(e)}"

def process_image_with_nova_arn(
    image_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this image content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process image using AWS Bedrock Nova Pro with ARN profile
    
    Args:
        image_path: Path to the image file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for image analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Image file validation
        if not os.path.exists(image_path):
            return f"Error: Image file not found: {image_path}"
        
        file_size = os.path.getsize(image_path)
        image_name = os.path.basename(image_path)
        
        print(f"Processing image: {image_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Image too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Read and encode image file
        print("Encoding image to base64...")
        with open(image_path, 'rb') as image_file:
            image_bytes = image_file.read()
            image_b64 = base64.b64encode(image_bytes).decode('utf-8')
        
        print(f"Image bytes length: {len(image_bytes)}")
        print(f"Base64 string length: {len(image_b64)}")
        
        # Check if image is too small (likely corrupted or test file)
        if len(image_bytes) < 1000:  # Less than 1KB
            print(f"Warning: Image file is very small ({len(image_bytes)} bytes). This may cause MIME type issues.")
            print("Consider using a larger, standard image file.")
        
        # Determine image format from file extension and validate
        file_ext = os.path.splitext(image_path)[1].lower()
        format_map = {
            '.jpg': 'jpeg',
            '.jpeg': 'jpeg',
            '.png': 'png',
            '.gif': 'gif',
            '.bmp': 'bmp',
            '.tiff': 'tiff',
            '.tif': 'tiff',
            '.webp': 'webp'
        }
        image_format = format_map.get(file_ext, 'jpeg')
        
        print(f"Detected image format: {image_format}")
        print(f"File extension: {file_ext}")
        
        # Validate image format
        if image_format not in ['jpeg', 'png', 'gif', 'webp']:
            print(f"Warning: Format {image_format} may not be fully supported by Nova Pro")
        
        # Prepare the message content
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": image_format,
                            "source": {
                                "bytes": image_b64
                            }
                        }
                    },
                    {"text": prompt}
                ]
            }
        ]
        
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Try using converse method first (more reliable for images)
        try:
            print("Trying converse method for image analysis...")
            response = bedrock.converse(
                modelId="amazon.nova-pro-v1:0",
                messages=messages,
                inferenceConfig=inference_config
            )
            
            # Extract response text
            result = response['output']['message']['content'][0]['text']
            
        except Exception as e:
            print(f"converse method failed: {str(e)}")
            print("Falling back to invoke_model method...")
            
            # Fallback to invoke_model method
            try:
                print("Trying invoke_model method...")
                response = bedrock.invoke_model(
                    modelId="amazon.nova-pro-v1:0",
                    body=json.dumps({
                        "messages": messages,
                        "inferenceConfig": inference_config
                    }),
                    contentType="application/json"
                )
                
                # Parse response
                response_body = json.loads(response['body'].read())
                result = response_body['output']['message']['content'][0]['text']
                
            except Exception as e2:
                print(f"invoke_model also failed: {str(e2)}")
                return f"Both API methods failed. Image may not be compatible with Nova Pro. Errors: {str(e)}, {str(e2)}"
        
        print("Image analysis completed successfully")
        return result
        
    except FileNotFoundError:
        return f"Error: Image file not found: {image_path}"
    except Exception as e:
        return f"Error processing image: {str(e)}"

def validate_arn_format(arn: str) -> bool:
    """Validate ARN format"""
    return arn.startswith("arn:aws:bedrock:") and ("inference-profile" in arn or "model-access-policy" in arn)

# Usage example
if __name__ == "__main__":
    # Configuration
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    video_file = "data/videos/appleq1.mp4"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    # Process video
    if os.path.exists(video_file):
        result = process_video_with_nova_arn(
            video_file, 
            profile_arn,
            "Analyze this video and provide key insights about the content, actions, and any notable elements."
        )
        print(f"\nVideo Analysis Result:")
        print("-" * 50)
        print(result)
    else:
        print(f"Video file not found: {video_file}")
    
    # Test image processing
    print("\n" + "="*60)
    print("TESTING IMAGE PROCESSING")
    print("="*60)
    
    image_file = "data/images/test_red_square.png"
    
    if os.path.exists(image_file):
        image_result = process_image_with_nova_arn(
            image_file,
            profile_arn,
            "Analyze this image and describe what you see in detail."
        )
        print(f"\nImage Analysis Result:")
        print("-" * 50)
        print(image_result)
    else:
        print(f"Image file not found: {image_file}")
        
        # Try to find any available images
        import glob
        available_images = glob.glob("data/images/*") + glob.glob("data/*.png") + glob.glob("data/*.jpg")
        if available_images:
            print(f"Available images found: {available_images}")
            # Use the first available image
            first_image = available_images[0]
            print(f"Testing with: {first_image}")
            
            image_result = process_image_with_nova_arn(
                first_image,
                profile_arn,
                "Analyze this image and describe what you see in detail."
            )
            print(f"\nImage Analysis Result:")
            print("-" * 50)
            print(image_result)
        else:
            print("No image files found in data directory")
        

Processing video: appleq1.mp4
File size: 3.71 MB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Encoding video to base64...
Video bytes length: 3885780
Base64 string length: 5181040
Base64 starts with: AAAAIGZ0eXBpc29tAAACAGlzb21pc28yYXZjMW1wNDEAAvS9bW...
Detected video format: mp4
File extension: .mp4
Sending request to Bedrock Nova Pro...
Trying invoke_model method...
Analysis completed successfully

Video Analysis Result:
--------------------------------------------------
The video is a financial chart analysis of Apple Inc. (AAPL) and the NASDAQ Composite Index (INTC) during the Q4 2020 earnings call. The chart displays the stock performance of both companies over a specified period, with AAPL showing a significant increase in value compared to INTC. The video highlights the positive financial results of Apple, indicating strong performance and growth. The chart includes various technical indicators such as moving averages

## AUDIO -ANALYSIS

In [3]:
import os
import asyncio
import base64
import json
import uuid
import pyaudio
import wave
from dotenv import load_dotenv
import boto3

# Load environment variables
load_dotenv()

# Audio configuration
INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000
CHANNELS = 1
FORMAT = pyaudio.paInt16
CHUNK_SIZE = 1024

class SimpleNovaSonic:
    def __init__(self, model_id='amazon.nova-sonic-v1:0', region='us-east-1', profile_arn=None):
        self.model_id = model_id
        self.region = region
        self.profile_arn = profile_arn
        self.client = None
        self.stream = None
        self.response = None
        self.is_active = False
        self.prompt_name = str(uuid.uuid4())
        self.content_name = str(uuid.uuid4())
        self.audio_content_name = str(uuid.uuid4())
        self.audio_queue = asyncio.Queue()
        self.role = None
        self.display_assistant_text = False
        
    def _initialize_client(self):
        """Initialize the Bedrock client with ARN profile support."""
        try:
            # Use boto3 client with ARN profile if provided
            if self.profile_arn:
                print(f"Using ARN profile: {self.profile_arn}")
                # For ARN profiles, we'll use the standard boto3 client
                self.client = boto3.client(
                    service_name='bedrock-runtime',
                    region_name=self.region,
                    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
                )
            else:
                # Fallback to environment credentials
                self.client = boto3.client(
                    service_name='bedrock-runtime',
                    region_name=self.region,
                    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
                )
            print("Bedrock client initialized successfully")
        except Exception as e:
            print(f"Error initializing Bedrock client: {e}")
            raise
    
    async def send_event(self, event_json):
        """Send an event to the stream."""
        try:
            event = {
                "event": json.loads(event_json)
            }
            # For boto3 client, we'll use invoke_model_with_response_stream
            # This is a simplified approach for testing
            print(f"Sending event: {event}")
        except Exception as e:
            print(f"Error sending event: {e}")
    
    async def start_session(self):
        """Start a new session with Nova Sonic."""
        if not self.client:
            self._initialize_client()
            
        print("Starting Nova Sonic session...")
        self.is_active = True
        
        # Send session start event
        session_start = '''
        {
          "event": {
            "sessionStart": {
              "inferenceConfiguration": {
                "maxTokens": 1024,
                "topP": 0.9,
                "temperature": 0.7
              }
            }
          }
        }
        '''
        await self.send_event(session_start)
        
        # Send prompt start event
        prompt_start = f'''
        {{
          "event": {{
            "promptStart": {{
              "promptName": "{self.prompt_name}",
              "textOutputConfiguration": {{
                "mediaType": "text/plain"
              }},
              "audioOutputConfiguration": {{
                "mediaType": "audio/lpcm",
                "sampleRateHertz": 24000,
                "sampleSizeBits": 16,
                "channelCount": 1,
                "voiceId": "matthew",
                "encoding": "base64",
                "audioType": "SPEECH"
              }}
            }}
          }}
        }}
        '''
        await self.send_event(prompt_start)
        
        # Send system prompt
        text_content_start = f'''
        {{
            "event": {{
                "contentStart": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}",
                    "type": "TEXT",
                    "interactive": true,
                    "role": "SYSTEM",
                    "textInputConfiguration": {{
                        "mediaType": "text/plain"
                    }}
                }}
            }}
        }}
        '''
        await self.send_event(text_content_start)
        
        system_prompt = "You are a friendly financial assistant. The user and you will engage in a spoken dialog " \
            "exchanging the transcripts of a natural real-time conversation. Keep your responses short, " \
            "generally two or three sentences for chatty scenarios."

        text_input = f'''
        {{
            "event": {{
                "textInput": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}",
                    "content": "{system_prompt}"
                }}
            }}
        }}
        '''
        await self.send_event(text_input)
        
        text_content_end = f'''
        {{
            "event": {{
                "contentEnd": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.content_name}"
                }}
            }}
        }}
        '''
        await self.send_event(text_content_end)
        
        print("Session started successfully")
    
    async def test_audio_file(self, audio_file_path):
        """Test audio file processing with Nova Sonic."""
        if not os.path.exists(audio_file_path):
            print(f"Audio file not found: {audio_file_path}")
            return
        
        print(f"Testing audio file: {audio_file_path}")
        
        # Read audio file
        try:
            with wave.open(audio_file_path, 'rb') as wav_file:
                # Get audio properties
                channels = wav_file.getnchannels()
                sample_width = wav_file.getsampwidth()
                frame_rate = wav_file.getframerate()
                n_frames = wav_file.getnframes()
                
                print(f"Audio properties:")
                print(f"  Channels: {channels}")
                print(f"  Sample width: {sample_width} bytes")
                print(f"  Frame rate: {frame_rate} Hz")
                print(f"  Duration: {n_frames / frame_rate:.2f} seconds")
                
                # Read audio data
                audio_data = wav_file.readframes(n_frames)
                
                # Convert to base64 for testing
                audio_b64 = base64.b64encode(audio_data).decode('utf-8')
                print(f"Audio data encoded to base64 (length: {len(audio_b64)})")
                
                # Simulate sending audio to Nova Sonic
                print("Simulating audio input to Nova Sonic...")
                
                # Start audio input
                await self.start_audio_input()
                
                # Send audio in chunks
                chunk_size = 1024
                for i in range(0, len(audio_data), chunk_size):
                    chunk = audio_data[i:i + chunk_size]
                    await self.send_audio_chunk(chunk)
                    await asyncio.sleep(0.1)  # Small delay between chunks
                
                # End audio input
                await self.end_audio_input()
                
                print("Audio file test completed")
                
        except Exception as e:
            print(f"Error processing audio file: {e}")
    
    async def start_audio_input(self):
        """Start audio input stream."""
        audio_content_start = f'''
        {{
            "event": {{
                "contentStart": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}",
                    "type": "AUDIO",
                    "interactive": true,
                    "role": "USER",
                    "audioInputConfiguration": {{
                        "mediaType": "audio/lpcm",
                        "sampleRateHertz": 16000,
                        "sampleSizeBits": 16,
                        "channelCount": 1,
                        "audioType": "SPEECH",
                        "encoding": "base64"
                    }}
                }}
            }}
        }}
        '''
        await self.send_event(audio_content_start)
        print("Audio input started")
    
    async def send_audio_chunk(self, audio_bytes):
        """Send an audio chunk to the stream."""
        if not self.is_active:
            return
            
        blob = base64.b64encode(audio_bytes)
        audio_event = f'''
        {{
            "event": {{
                "audioInput": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}",
                    "content": "{blob.decode('utf-8')}"
                }}
            }}
        }}
        '''
        await self.send_event(audio_event)
    
    async def end_audio_input(self):
        """End audio input stream."""
        audio_content_end = f'''
        {{
            "event": {{
                "contentEnd": {{
                    "promptName": "{self.prompt_name}",
                    "contentName": "{self.audio_content_name}"
                }}
            }}
        }}
        '''
        await self.send_event(audio_content_end)
        print("Audio input ended")
    
    async def end_session(self):
        """End the session."""
        if not self.is_active:
            return
            
        prompt_end = f'''
        {{
            "event": {{
                "promptEnd": {{
                    "promptName": "{self.prompt_name}"
                }}
            }}
        }}
        '''
        await self.send_event(prompt_end)
        
        session_end = '''
        {
            "event": {
                "sessionEnd": {}
            }
        }
        '''
        await self.send_event(session_end)
        
        self.is_active = False
        print("Session ended")

async def main():
    # Your ARN profile
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-sonic-v1:0"
    
    # Create Nova Sonic client
    nova_client = SimpleNovaSonic(
        model_id='amazon.nova-sonic-v1:0',
        region='us-east-1',
        profile_arn=profile_arn
    )
    
    try:
        # Start session
        await nova_client.start_session()
        
        # Test with the audio file we created
        audio_file = "data/audio/test_tone.wav"
        await nova_client.test_audio_file(audio_file)
        
        # Wait a bit for processing
        await asyncio.sleep(2)
        
    except Exception as e:
        print(f"Error in main: {e}")
    finally:
        # End session
        await nova_client.end_session()
        print("Test completed")

if __name__ == "__main__":
    print("Testing Nova Sonic with ARN Profile")
    print("=" * 50)
    
    # Check if required packages are installed
    try:
        import pyaudio
        print("✓ PyAudio is available")
    except ImportError:
        print("✗ PyAudio not found. Install with: pip install pyaudio")
        exit(1)
    
    try:
        import boto3
        print("✓ Boto3 is available")
    except ImportError:
        print("✗ Boto3 not found. Install with: pip install boto3")
        exit(1)
    
    # Check environment variables
    if os.getenv('AWS_ACCESS_KEY_ID') and os.getenv('AWS_SECRET_ACCESS_KEY'):
        print("✓ AWS credentials found in environment")
    else:
        print("✗ AWS credentials not found. Check your .env file")
        exit(1)
    
    print("\nStarting Nova Sonic test...")
    asyncio.run(main())


Testing Nova Sonic with ARN Profile
✓ PyAudio is available
✓ Boto3 is available
✓ AWS credentials found in environment

Starting Nova Sonic test...


RuntimeError: asyncio.run() cannot be called from a running event loop